In [38]:
import numpy as np
import timeit

In [39]:
#A) Genera muestras de la distribucion normal sumando 12 uniformes
def normal_12_uniformes(mu, var, n):
    """
    Aproxima N(mu, var) sumando 12 variables U(0,1).
    """
    sigma = np.sqrt(var)
    u = np.random.uniform(0.0, 1.0, size=(n, 12))
    z = np.sum(u, axis=1) - 6
    return mu + sigma * z
 

In [40]:
def normal_rechazo(mu, var, n):
    """
    Genera muestras de N(mu, var) usando muestreo con rechazo.
    """
    sigma = np.sqrt(var)
    L = 6.0
    M = 2 * L / np.sqrt(2 * np.pi)
 
    muestras = []
    total = 0
 
    while total < n:
        batch_size = max(10000, n)
        y = np.random.uniform(-L, L, size=batch_size)
        u = np.random.uniform(0.0, 1.0, size=batch_size)
 
        # f(y) = densidad normal estándar en y
        fy = (1.0 / np.sqrt(2.0 * np.pi)) * np.exp(-(y ** 2) / 2.0)
 
        # g(y) = densidad uniforme en y
        gy = 1.0 / (2.0 * L)
 
        aceptadas = y[u <= fy / (M * gy)]
 
        if len(aceptadas) > 0:
            muestras.append(aceptadas)
            total += len(aceptadas)
 
    muestras = np.concatenate(muestras)[:n]
    return mu + sigma * muestras

In [41]:
def normal_box_muller(mu, var, n):
    """
    Genera muestras de N(mu, var) usando la transformación de Box-Muller.
    """
    sigma = np.sqrt(var)
    u1 = np.random.uniform(0.0, 1.0, size=n)
    u2 = np.random.uniform(0.0, 1.0, size=n)
 
    z = np.cos(2 * np.pi * u1) * np.sqrt(-2 * np.log(u2))
    return mu + sigma * z

In [42]:
def comparar_tiempos(mu=0, var=1, n=10000, repeticiones=3):
    """Compara tiempos de ejecución de los tres métodos versus numpy."""
    t1 = timeit.timeit(lambda: normal_12_uniformes(mu, var, n), number=repeticiones)
    t2 = timeit.timeit(lambda: normal_rechazo(mu, var, n), number=repeticiones)
    t3 = timeit.timeit(lambda: normal_box_muller(mu, var, n), number=repeticiones)
    t4 = timeit.timeit(lambda: np.random.normal(loc=mu, scale=np.sqrt(var), size=n), number=repeticiones)
 
    print(f"n = {n}, repeticiones = {repeticiones}")
    print(f"1) 12 uniformes       : {t1:.6f} s")
    print(f"2) Rechazo            : {t2:.6f} s")
    print(f"3) Box-Muller         : {t3:.6f} s")
    print(f"4) numpy.random.normal: {t4:.6f} s")

In [43]:
if __name__ == "__main__":
    mu = 5
    var = 4
    n = 10000
 
    x1 = normal_12_uniformes(mu, var, n)
    x2 = normal_rechazo(mu, var, n)
    x3 = normal_box_muller(mu, var, n)
    x4 = np.random.normal(loc=mu, scale=np.sqrt(var), size=n)
 
    print("Medias aproximadas (esperado: 5.0):")
    print("  12 uniformes :", np.mean(x1))
    print("  Rechazo      :", np.mean(x2))
    print("  Box-Muller   :", np.mean(x3))
    print("  NumPy        :", np.mean(x4))
 
    print("\nVarianzas aproximadas (esperado: 4.0):")
    print("  12 uniformes :", np.var(x1))
    print("  Rechazo      :", np.var(x2))
    print("  Box-Muller   :", np.var(x3))
    print("  NumPy        :", np.var(x4))
 
    print("\nComparación de tiempos:")
    comparar_tiempos(mu=mu, var=var, n=10000, repeticiones=3)

Medias aproximadas (esperado: 5.0):
  12 uniformes : 5.001537558184482
  Rechazo      : 4.967369969731716
  Box-Muller   : 5.031795064099782
  NumPy        : 4.98713719293094

Varianzas aproximadas (esperado: 4.0):
  12 uniformes : 3.9081649864554566
  Rechazo      : 4.040839161119756
  Box-Muller   : 4.026022631664548
  NumPy        : 3.938643476880333

Comparación de tiempos:
n = 10000, repeticiones = 3
1) 12 uniformes       : 0.007999 s
2) Rechazo            : 0.010023 s
3) Box-Muller         : 0.002208 s
4) numpy.random.normal: 0.001489 s
